# WoRMS + OBIS + Copernicus — Esplorazione per il Capstone

Obiettivo: verificare, per le 200 specie selezionate dal notebook `01_orca_dataset_exploration`, quante sono effettivamente **risolvibili tassonomicamente** (WoRMS) e quante hanno **dati di occorrenza geolocalizzati** (OBIS) — prima di costruire il modulo di analisi ambientale sopra dati che potrebbero non esistere per alcune classi.

**Attenzione preliminare:** alcune categorie ORCA sono etichette generiche a livello di gruppo (es. "fish", "coral", "starfish") invece di specie vere e proprie — per queste il `genus`/`species` in `data.json` è `None`, quindi non sono risolvibili su WoRMS/OBIS per nome scientifico. Questo notebook le identifica esplicitamente invece di fallire silenziosamente.

> Nota implementativa: il codice segue la documentazione ufficiale delle API (OBIS v3, WoRMS REST) ma richiede un ambiente con accesso di rete per essere eseguito.

## 0. Setup

In [1]:
# pip install requests pandas tqdm --quiet

import json
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_rows", 50)

WORMS_BASE = "https://www.marinespecies.org/rest"
OBIS_BASE = "https://api.obis.org/v3"

SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json"})
REQUEST_DELAY = 0.3  # secondi tra una chiamata e l'altra, per non sovraccaricare le API pubbliche

## 1. Ricostruzione della lista delle 200 specie selezionate

Stessa logica del notebook 01 (soglia 20 immagini/specie sullo split `intra_class_train`), ma qui teniamo anche `genus`/`species` per la risoluzione tassonomica. Aggiusta `ORCA_DIR` se il percorso locale è diverso.

In [2]:
def find_project_root(marker=".gitignore"):
    """Risale le cartelle da cwd finche' non trova il marker della root del progetto.
    Funziona indipendentemente da dove si trova fisicamente questo notebook
    (notebooks/, root, o altre sottocartelle) e da dove viene lanciato il kernel."""
    path = Path.cwd()
    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent
    raise FileNotFoundError(f"Marker '{marker}' non trovato risalendo da {Path.cwd()}")

PROJECT_ROOT = find_project_root()
ORCA_DIR = PROJECT_ROOT / "data" / "orca"
SPLIT_DIR = ORCA_DIR / "split_annotations"
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)
MIN_IMAGES_PER_SPECIES = 20

print("Project root:", PROJECT_ROOT)
print("Split dir:", SPLIT_DIR)
print("Docs dir:", DOCS_DIR)

with open(SPLIT_DIR / "intra_class_train.json") as f:
    train_coco = json.load(f)

categories_df = pd.DataFrame(train_coco["categories"]).rename(columns={"id": "category_id"})
categories_df["common_name"] = categories_df["name"]

annotations_df = pd.DataFrame(train_coco["annotations"])
annotations_df = annotations_df[annotations_df["label"].isin([0, 2])]

counts = annotations_df.merge(categories_df, on="category_id").groupby("common_name")["image_id"].nunique()
selected_species = counts[counts >= MIN_IMAGES_PER_SPECIES].index.tolist()

species_df = categories_df[categories_df["common_name"].isin(selected_species)].drop_duplicates(subset="common_name").copy()
species_df["has_binomial"] = species_df["genus"].notna() & species_df["species"].notna()
species_df["binomial"] = (species_df["genus"].fillna("") + " " + species_df["species"].fillna("")).str.strip()

print(f"Specie selezionate: {len(species_df)}")
print(f"Con binomio scientifico (genus+species): {species_df['has_binomial'].sum()}")
print(f"Senza binomio (etichette generiche, es. 'fish', 'coral'): {(~species_df['has_binomial']).sum()}")
print("\nEsempi di etichette generiche non risolvibili:")
print(species_df.loc[~species_df["has_binomial"], "common_name"].tolist())

Project root: /home/ubuntu/marine_ai
Split dir: /home/ubuntu/marine_ai/data/orca/split_annotations
Docs dir: /home/ubuntu/marine_ai/docs
Specie selezionate: 200
Con binomio scientifico (genus+species): 161
Senza binomio (etichette generiche, es. 'fish', 'coral'): 39

Esempi di etichette generiche non risolvibili:
['moorish idol', 'oyster', 'dragonfish', 'bichir', 'unicornfish', 'anglerfish', 'serranidae', 'basking shark', 'pegasus sea moth', 'mobula ray', 'basslet', 'marlin', 'port jackson shark', 'pederson cleaner shrimp', 'leatherback sea turtle', 'african lungfish', 'australian lungfish', 'frilled shark', 'basket star', 'pike', 'trachemys scripta elegans', 'pistol shrimp', 'spiny lobster', 'squat lobster', 'kingfish', 'surgeonfish', 'porcelain crab', 'trumpetfish', 'sleeper shark', 'dottyback', 'perch', 'bubble coral', 'fangtooth fish', 'dragonet', 'catfish', 'bamboo shark', 'manta ray', 'bonito tuna', 'fiddler crab']


## 2. Risoluzione tassonomica su WoRMS

Per ogni binomio, chiamiamo `AphiaRecordsByName` per ottenere l'AphiaID, il nome scientifico accettato (se il binomio in ORCA è un sinonimo) e la classificazione completa. Interroghiamo solo le specie con binomio valido dalla cella precedente.

In [3]:
def worms_lookup(scientific_name: str) -> dict | None:
    """Interroga AphiaRecordsByName e restituisce il primo match, o None se non trovato."""
    url = f"{WORMS_BASE}/AphiaRecordsByName/{quote(scientific_name)}"
    params = {"like": "false", "marine_only": "true"}
    try:
        resp = SESSION.get(url, params=params, timeout=15)
        if resp.status_code == 204 or not resp.text.strip():
            return None
        resp.raise_for_status()
        data = resp.json()
        return data[0] if isinstance(data, list) and data else None
    except requests.RequestException as e:
        return {"error": str(e)}

worms_results = {}
to_lookup = species_df[species_df["has_binomial"]][["common_name", "binomial"]]

for _, row in tqdm(to_lookup.iterrows(), total=len(to_lookup)):
    worms_results[row["common_name"]] = worms_lookup(row["binomial"])
    time.sleep(REQUEST_DELAY)

n_resolved = sum(1 for v in worms_results.values() if v and "error" not in v)
n_errors = sum(1 for v in worms_results.values() if v and "error" in v)
n_not_found = sum(1 for v in worms_results.values() if v is None)

print(f"Risolte su WoRMS: {n_resolved}")
print(f"Non trovate (probabile sinonimo non standard o nome errato): {n_not_found}")
print(f"Errori di rete/API: {n_errors}")

  0%|          | 0/161 [00:00<?, ?it/s]

Risolte su WoRMS: 147
Non trovate (probabile sinonimo non standard o nome errato): 14
Errori di rete/API: 0


In [4]:
def extract_worms_fields(result: dict | None) -> dict:
    if not result or "error" in result:
        return {"aphia_id": None, "accepted_name": None, "worms_status": "not_found" if result is None else "error"}
    return {
        "aphia_id": result.get("AphiaID"),
        "accepted_name": result.get("valid_name") or result.get("scientificname"),
        "worms_status": result.get("status"),
    }

worms_df = pd.DataFrame([
    {"common_name": name, **extract_worms_fields(res)}
    for name, res in worms_results.items()
])

species_df = species_df.merge(worms_df, on="common_name", how="left")
species_df.head(10)

,category_id,name,supercategory,kingdom,phylum,class,order,family,genus,species,common_name,has_binomial,binomial,aphia_id,accepted_name,worms_status
0,272,fish,fish,Animalia,Chordata,Teleostei,Tetraodontiformes,Balistidae,Abalistes,stellaris,fish,True,Abalistes stellaris,219872.0,Abalistes stellatus,unaccepted
1,190,cymbiola vespertilio,cymbiola vespertilio,Animalia,Mollusca,Gastropoda,Neogastropoda,Volutidae,Cymbiola,vespertilio,cymbiola vespertilio,True,Cymbiola vespertilio,568006.0,Cymbiola vespertilio,accepted
2,559,snake eel,snake eel,Animalia,Chordata,Teleostei,Anguilliformes,Ophichthidae,Callechelys,bilinearis,snake eel,True,Callechelys bilinearis,275464.0,Callechelys bilinearis,accepted
3,440,panulirus argus,panulirus argus,Animalia,Arthropoda,Malacostraca,Decapoda,Palinuridae,Panulirus,argus,panulirus argus,True,Panulirus argus,382891.0,Panulirus argus,accepted
4,406,needlefish,needlefish,Animalia,Chordata,Teleostei,Beloniformes,Belonidae,Ablennes,hians,needlefish,True,Ablennes hians,159246.0,Ablennes hians,accepted
5,397,moorish idol,moorish idol,Animalia,Chordata,Teleostei,Acanthuriformes,Zanclidae,NaN,NaN,moorish idol,False,,NaN,NaN,NaN
6,164,coral,coral,Animalia,Cnidaria,Octocorallia,Scleralcyonacea,Keratoisididae,Acanella,arbuscula,coral,True,Acanella arbuscula,125371.0,Acanella arbuscula,accepted
7,12,amberjack,amberjack,Animalia,Chordata,Teleostei,Carangiformes,Carangidae,Seriola,dumerili,amberjack,True,Seriola dumerili,126816.0,Seriola dumerili,accepted
8,433,oyster,oyster,Animalia,Mollusca,Bivalvia,Pectinida,Anomiidae,Anomia,NaN,oyster,False,Anomia,NaN,NaN,NaN
9,225,dragonfish,dragonfish,Animalia,Chordata,Teleostei,Stomiiformes,Stomiidae,Bathophilus,NaN,dragonfish,False,Bathophilus,NaN,NaN,NaN


## 3. Copertura occorrenze su OBIS

Per ogni specie risolta (con AphiaID valido), interroghiamo `occurrence` per contare quante osservazioni geolocalizzate esistono. Usiamo `size=1` perché in questa fase ci interessa solo il conteggio totale (campo `total` nella risposta), non ancora i record completi.

In [5]:
def obis_occurrence_count(aphia_id: int) -> int | None:
    url = f"{OBIS_BASE}/occurrence"
    params = {"taxonid": aphia_id, "size": 1}
    try:
        resp = SESSION.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        return data.get("total")
    except requests.RequestException:
        return None

obis_counts = {}
resolvable = species_df[species_df["aphia_id"].notna()]

for _, row in tqdm(resolvable.iterrows(), total=len(resolvable)):
    obis_counts[row["common_name"]] = obis_occurrence_count(int(row["aphia_id"]))
    time.sleep(REQUEST_DELAY)

obis_df = pd.DataFrame(list(obis_counts.items()), columns=["common_name", "obis_occurrence_count"])
species_df = species_df.merge(obis_df, on="common_name", how="left")

  0%|          | 0/147 [00:00<?, ?it/s]

## 4. Report di copertura complessivo

Questo è il numero chiave per la sinossi: quante delle 200 classi del classificatore possono *anche* essere usate nel modulo di analisi ambientale/distribuzione.

In [6]:
species_df["env_module_ready"] = species_df["obis_occurrence_count"].fillna(0) > 0

print(f"Specie totali selezionate per il classificatore: {len(species_df)}")
print(f"  - senza binomio (etichetta generica): {(~species_df['has_binomial']).sum()}")
print(f"  - con binomio ma non risolte su WoRMS: {(species_df['has_binomial'] & species_df['aphia_id'].isna()).sum()}")
print(f"  - risolte su WoRMS ma 0 occorrenze OBIS: {(species_df['aphia_id'].notna() & (species_df['obis_occurrence_count'].fillna(0) == 0)).sum()}")
print(f"  - pienamente utilizzabili anche nel modulo ambientale: {species_df['env_module_ready'].sum()}")

print("\nTop 15 specie per numero di occorrenze OBIS (le più 'ricche' di dati di distribuzione):")
species_df.sort_values("obis_occurrence_count", ascending=False)[["common_name", "binomial", "accepted_name", "obis_occurrence_count"]].head(15)

Specie totali selezionate per il classificatore: 200
  - senza binomio (etichetta generica): 39
  - con binomio ma non risolte su WoRMS: 14
  - risolte su WoRMS ma 0 occorrenze OBIS: 0
  - pienamente utilizzabili anche nel modulo ambientale: 147

Top 15 specie per numero di occorrenze OBIS (le più 'ricche' di dati di distribuzione):


,common_name,binomial,accepted_name,obis_occurrence_count
174,american lobster,Homarus americanus,Homarus americanus,921016.0
131,eudyptula minor,Eudyptula minor,Eudyptula minor,557895.0
40,strongylocentrotus franciscanus,Strongylocentrotus franciscanus,Mesocentrotus franciscanus,378611.0
42,chelonia mydas,Chelonia mydas,Chelonia mydas,243974.0
135,green sea turtle,Chelonia mydas,Chelonia mydas,243974.0
50,bull shark,Carcharhinus leucas,Carcharhinus leucas,236322.0
127,blacktip reef shark,Carcharhinus melanopterus,Carcharhinus melanopterus,226964.0
187,pygoscelis adeliae,Pygoscelis adeliae,Pygoscelis adeliae,189836.0
138,atlantic blue crab,Callinectes sapidus,Callinectes sapidus,189187.0
62,european green crab,Carcinus maenas,Carcinus maenas,188042.0


## 4bis. Fusione classi duplicate (stesso AphiaID, common_name diversi)

Verifica visiva sulle immagini (fatta a mano sul notebook 01) ha confermato che alcune coppie di `common_name` diverse sono in realta' lo stesso animale (nome comune + nome scientifico usato come label separata), mentre altre — pur condividendo l'AphiaID risolto da WoRMS — sono animali morfologicamente distinti, segno di un binomio errato nei metadati originali ORCA per uno dei due.

**Fuse (stesso animale):**
- `chelonia mydas` + `green sea turtle` -> `green sea turtle`
- `peacock mantis shrimp` + `odontodactylus scyllarus` -> `peacock mantis shrimp`
- `nudibranch` + `sea slug` -> `nudibranch`
- `seadragon` + `leafy sea dragon` -> `leafy sea dragon`

**Tenute separate nonostante l'AphiaID condiviso (animali diversi, binomio ORCA da correggere):**
- `seahorse` / `pipefish` — genn diversi (Syngnathidae), il match WoRMS e' un falso positivo da investigare a parte, non una fusione.

In [7]:
CANONICAL_LABEL_MAP = {
    "chelonia mydas": "green sea turtle",
    "odontodactylus scyllarus": "peacock mantis shrimp",
    "sea slug": "nudibranch",
    "seadragon": "leafy sea dragon",
}

species_df["canonical_label"] = species_df["common_name"].replace(CANONICAL_LABEL_MAP)

# per le classi fuse, sommiamo le occorrenze OBIS e teniamo l'AphiaID/accepted_name comune
canonical_df = (
    species_df
    .groupby("canonical_label", as_index=False)
    .agg({
        "aphia_id": "first",
        "accepted_name": "first",
        "has_binomial": "max",
        "obis_occurrence_count": "sum",
        "env_module_ready": "max",
        "common_name": lambda x: list(x),  # teniamo traccia dei common_name originali fusi
    })
    .rename(columns={"common_name": "merged_from"})
)

print(f"Classi prima della fusione: {species_df['common_name'].nunique()}")
print(f"Classi dopo la fusione: {canonical_df.shape[0]}")

merged_rows = canonical_df[canonical_df["merged_from"].apply(len) > 1]
print(f"\nClassi risultanti da una fusione ({len(merged_rows)}):")
merged_rows[["canonical_label", "merged_from", "obis_occurrence_count"]]

Classi prima della fusione: 200
Classi dopo la fusione: 196

Classi risultanti da una fusione (4):


,canonical_label,merged_from,obis_occurrence_count
95,green sea turtle,"[chelonia mydas, green sea turtle]",487948.0
105,leafy sea dragon,"[seadragon, leafy sea dragon]",2430.0
126,nudibranch,"[nudibranch, sea slug]",5568.0
138,peacock mantis shrimp,"[peacock mantis shrimp, odontodactylus scyllarus]",3310.0


## 4ter. Diagnosi seahorse / pipefish

Prima di correggere il binomio, guardiamo cosa c'e' davvero nei metadati grezzi per capire dove si e' rotta la risoluzione.

In [8]:
diagnostic_cols = ["common_name", "genus", "species", "binomial", "has_binomial", "family", "order", "aphia_id", "accepted_name"]
species_df[species_df["common_name"].isin(["seahorse", "pipefish"])][diagnostic_cols]

,common_name,genus,species,binomial,has_binomial,family,order,aphia_id,accepted_name
71,seahorse,Acentronura,dendritica,Acentronura dendritica,True,Syngnathidae,Syngnathiformes,159442.0,Amphelikturus dendriticus
91,pipefish,Acentronura,dendritica,Acentronura dendritica,True,Syngnathidae,Syngnathiformes,159442.0,Amphelikturus dendriticus


**Diagnosi confermata:** `seahorse` e `pipefish` hanno lo stesso `genus`/`species` (`Acentronura dendritica`) nei metadati originali ORCA — un errore di data entry a monte (probabile confusione tra le due categorie durante l'annotazione), non una particolarita' biologica reale. Le immagini mostrano animali morfologicamente diversi (verificato visivamente), quindi il binomio condiviso non e' affidabile per nessuna delle due classi.

**Decisione per il capstone (v1):** le due classi restano nel classificatore CV (si basa sulle immagini, non sul binomio), ma vengono escluse dal modulo ambientale/di distribuzione perche' non abbiamo un nome scientifico affidabile su cui interrogare WoRMS/OBIS. Non tentiamo una correzione manuale del genere per restare nello scope fattibile del progetto.

**Nota per una v2.0 futura (fuori scope capstone):** con piu' tempo, un biologo potrebbe rivedere un campione di immagini di entrambe le classi e assegnare manualmente il genere corretto (es. `Hippocampus` per i cavallucci, un genere di pesce ago per l'altra), poi ri-lanciare la risoluzione WoRMS/OBIS solo su quelle due righe.

In [9]:
UNRELIABLE_TAXONOMY = ["seahorse", "pipefish"]

species_df["taxonomy_reliable"] = ~species_df["common_name"].isin(UNRELIABLE_TAXONOMY)

# il modulo ambientale richiede sia occorrenze OBIS sia un binomio affidabile
species_df["env_module_ready"] = (
    (species_df["obis_occurrence_count"].fillna(0) > 0) & species_df["taxonomy_reliable"]
)

print(f"Classi escluse dal modulo ambientale per tassonomia inaffidabile: {(~species_df['taxonomy_reliable']).sum()}")
print(f"Classi pienamente utilizzabili nel modulo ambientale (aggiornato): {species_df['env_module_ready'].sum()}")

# canonical_df (sezione 4bis) e' stato costruito prima di questa correzione: riallineiamo
canonical_df = canonical_df.merge(
    species_df[species_df["common_name"].isin(UNRELIABLE_TAXONOMY)][["common_name", "env_module_ready"]]
    .rename(columns={"common_name": "canonical_label", "env_module_ready": "_env_ready_fix"}),
    on="canonical_label", how="left",
)
canonical_df["env_module_ready"] = canonical_df["_env_ready_fix"].combine_first(canonical_df["env_module_ready"])
canonical_df = canonical_df.drop(columns="_env_ready_fix")

Classi escluse dal modulo ambientale per tassonomia inaffidabile: 2
Classi pienamente utilizzabili nel modulo ambientale (aggiornato): 145


## 5. Scouting Copernicus Marine (variabili ambientali)

Copernicus Marine Service richiede una registrazione gratuita (account su https://data.marine.copernicus.eu) e il toolbox ufficiale `copernicusmarine`. Non è interrogabile in modo anonimo come OBIS/WoRMS, quindi qui documentiamo solo la forma della chiamata: va completata con le credenziali quando verra' costruito il modulo ambientale vero e proprio.

```python
# pip install copernicusmarine --quiet
import copernicusmarine

# prima esecuzione: richiede login interattivo (salva le credenziali in locale)
# copernicusmarine.login()

# esempio: subset di temperatura superficiale del mare in un bounding box e intervallo temporale
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",  # verificare l'ID prodotto corretto per la variabile/area scelta
    variables=["thetao"],  # temperatura potenziale
    minimum_longitude=..., maximum_longitude=...,
    minimum_latitude=..., maximum_latitude=...,
    start_datetime="2024-01-01", end_datetime="2024-12-31",
)
```

Il flusso naturale per il modulo ambientale sarà: per ogni occorrenza OBIS (lat/lon/data), interrogare Copernicus per la temperatura/salinità/clorofilla nello stesso punto e periodo, per costruire il dataset di correlazione avvistamento-condizioni ambientali.

## 6. Export per PPT/sinossi

In [10]:
species_df.to_csv(DOCS_DIR / "species_environmental_coverage.csv", index=False)
canonical_df.to_csv(DOCS_DIR / "species_canonical_mapping.csv", index=False)

with open(DOCS_DIR / "canonical_label_map.json", "w") as f:
    json.dump(CANONICAL_LABEL_MAP, f, indent=2, ensure_ascii=False)
print("Salvato canonical_label_map.json — riusalo nel notebook di training per rinominare le label prima del fine-tuning")

summary = {
    "n_specie_selezionate_classificatore_pre_fusione": len(species_df),
    "n_specie_dopo_fusione_duplicati": int(canonical_df.shape[0]),
    "n_classi_fuse": int((canonical_df["merged_from"].apply(len) > 1).sum()),
    "n_classi_escluse_tassonomia_inaffidabile": int((~species_df["taxonomy_reliable"]).sum()),
    "n_senza_binomio": int((~species_df["has_binomial"]).sum()),
    "n_risolte_worms": int(species_df["aphia_id"].notna().sum()),
    "n_con_occorrenze_obis_e_tassonomia_affidabile": int(species_df["env_module_ready"].sum()),
    "totale_occorrenze_obis": int(species_df["obis_occurrence_count"].fillna(0).sum()),
}

with open(DOCS_DIR / "environmental_coverage_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

summary

Salvato canonical_label_map.json — riusalo nel notebook di training per rinominare le label prima del fine-tuning


{'n_specie_selezionate_classificatore_pre_fusione': 200,
 'n_specie_dopo_fusione_duplicati': 196,
 'n_classi_fuse': 4,
 'n_classi_escluse_tassonomia_inaffidabile': 2,
 'n_senza_binomio': 39,
 'n_risolte_worms': 147,
 'n_con_occorrenze_obis_e_tassonomia_affidabile': 145,
 'totale_occorrenze_obis': 4856944}